# ⚙️ Terminal 1 — Faust Streams Processor
Consumes `raw-data`, runs ML prediction on each record, publishes result to `predictions`.

**Run this FIRST** before the producer or consumer.

In [ ]:
!pip install faust-streaming confluent-kafka scikit-learn joblib --quiet

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ════════════════════════════════════════════════
#   FILL IN YOUR CONFLUENT CLOUD CREDENTIALS HERE
# ════════════════════════════════════════════════
BOOTSTRAP_SERVERS = 'pkc-XXXXX.us-east-1.aws.confluent.cloud:9092'  # <-- replace
SASL_USERNAME     = 'YOUR_API_KEY'                                    # <-- replace
SASL_PASSWORD     = 'YOUR_API_SECRET'                                 # <-- replace
MODEL_PATH        = '/content/drive/MyDrive/kafka-assignment/bike_model.joblib'
# ════════════════════════════════════════════════

In [ ]:
# Write streams_app.py using individual lines — avoids all f-string escaping issues
lines = [
    'import faust',
    'import joblib',
    'import json',
    'import time',
    'import ssl',
    'import numpy as np',
    '',
    f'BOOTSTRAP  = "{BOOTSTRAP_SERVERS}"',
    f'API_KEY    = "{SASL_USERNAME}"',
    f'API_SECRET = "{SASL_PASSWORD}"',
    f'MODEL_FILE = "{MODEL_PATH}"',
    '',
    'artifact = joblib.load(MODEL_FILE)',
    'model    = artifact["model"]',
    'FEATURES = artifact["features"]',
    'print("[PROCESSOR] Model loaded. Features: " + str(FEATURES))',
    '',
    'ssl_context = ssl.create_default_context()',
    '',
    'app = faust.App(',
    '    "bike-streams-processor",',
    '    broker="kafka://" + BOOTSTRAP,',
    '    broker_credentials=faust.SASLCredentials(',
    '        username=API_KEY,',
    '        password=API_SECRET,',
    '        mechanism="PLAIN",',
    '        ssl_context=ssl_context,',
    '    ),',
    '    value_serializer="raw",',
    ')',
    '',
    'raw_topic         = app.topic("raw-data",    value_type=bytes)',
    'predictions_topic = app.topic("predictions", value_type=bytes)',
    '',
    '@app.agent(raw_topic)',
    'async def process_stream(stream):',
    '    async for raw_bytes in stream:',
    '        try:',
    '            record = json.loads(raw_bytes)',
    '            feature_values = [record.get(f, 0) for f in FEATURES]',
    '            X = np.array(feature_values).reshape(1, -1)',
    '            prediction = float(model.predict(X)[0])',
    '            actual     = record.get("cnt", None)',
    '            output = {',
    '                "row_index"       : record.get("row_index"),',
    '                "timestamp"       : record.get("timestamp"),',
    '                "processed_at"    : time.time(),',
    '                "predicted_count" : round(prediction, 1),',
    '                "actual_count"    : actual,',
    '                "season"          : record.get("season"),',
    '                "hr"              : record.get("hr"),',
    '                "temp"            : record.get("temp"),',
    '                "weathersit"      : record.get("weathersit"),',
    '            }',
    '            await predictions_topic.send(value=json.dumps(output).encode())',
    '            row_idx = record.get("row_index")',
    '            pred_r  = round(prediction, 1)',
    '            print("[PROCESSED] row=" + str(row_idx) + " | pred=" + str(pred_r) + " | actual=" + str(actual))',
    '        except Exception as e:',
    '            print("[ERROR] " + str(e))',
]

with open('/content/streams_app.py', 'w') as f:
    f.write('\n'.join(lines))

print('streams_app.py written successfully!')
print('--- Preview first 10 lines ---')
with open('/content/streams_app.py') as f:
    for i, line in enumerate(f.readlines()[:10]):
        print(line, end='')

In [ ]:
# Start the Faust worker — this cell runs continuously (that's expected!)
!faust -A streams_app worker -l info